# Parte 4 — Treinamento e validação de modelos

Nesta parte comparamos alguns algoritmos de classificação para escolher
qual leva para a etapa de otimização.

O conjunto de teste (salvo na
Parte 3, em `data/model_input/test.parquet`) não é usado aqui. Ele só
vai ser tocado mais para frente, depois de já termos
escolhido e otimizado o modelo. Se usássemos o teste repetidamente para
comparar modelos e ajustar hiperparâmetros, estaríamos "vazando" ele aos
poucos — o teste deixaria de medir a generalização real e passaria a
medir o quão bem o modelo decorou aquele conjunto específico.

Em vez disso, comparamos os modelos com **cross-validation dentro do
próprio treino**: dividimos o treino em pedaços (folds), treinamos em uns
e validamos em outro, repetindo — assim cada modelo é avaliado várias
vezes em dados que ele não viu, sem gastar o teste.


## 1. Setup

In [1]:
import sys
from pathlib import Path

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.pipeline import Pipeline

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.utils.paths import GOLD_DIR
from src.preprocessing.pipeline import build_preprocessor, COLUNAS_CATEGORICAS, COLUNAS_NUMERICAS

PASTA_MODELO = GOLD_DIR.parent / "model_input"


## 2. Carregar o treino (o mesmo split salvo na Parte 3)

Reaproveitamos exatamente o `train.parquet` gerado na Parte 3 — mesma
seed, mesmo split, ninguém precisa refazer a divisão.


In [2]:
train = pd.read_parquet(PASTA_MODELO / "train.parquet")

X_train = train[COLUNAS_CATEGORICAS + COLUNAS_NUMERICAS]
y_train = train["alfabetizado_flag"]

print(f"{len(X_train):,} linhas de treino")
y_train.value_counts(normalize=True).round(3)


3,094,399 linhas de treino


alfabetizado_flag
1    0.513
0    0.487
Name: proportion, dtype: float64

## 3. Definir os modelos a comparar

Dois algoritmos com perfis bem diferentes:

- **Regressão Logística**: modelo linear, rápido, fácil de interpretar —
  serve de baseline.
- **Random Forest**: ensemble de árvores, captura relações não-lineares
  e interações entre variáveis que a Regressão Logística não vê.

Cada um entra num `Pipeline` completo junto com o `preprocessor` da Parte
3 — pré-processamento e modelo são uma coisa só, reajustados a cada fold.


In [3]:
modelos = {
    "Regressao Logistica": LogisticRegression(max_iter=200, random_state=42),
    "Random Forest": RandomForestClassifier(
        n_estimators=100, max_depth=None, n_jobs=-1, random_state=42
    ),
}

metricas = ["accuracy", "precision", "recall", "f1", "roc_auc"]


## 4. Rodar a validação cruzada

`cv=3` (3 folds) — um bom equilíbrio entre robustez da estimativa e tempo
de execução, considerando o volume de dados. Isso pode levar alguns
minutos, principalmente na Random Forest.


In [4]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

resultados = {}

for nome, modelo in modelos.items():
    pipeline = Pipeline(steps=[
        ("preprocessor", build_preprocessor()),
        ("modelo", modelo),
    ])

    print(f"Rodando cross-validation: {nome}...")
    scores = cross_validate(
        pipeline, X_train, y_train,
        cv=cv, scoring=metricas, n_jobs=1,
    )
    resultados[nome] = {
        metrica: scores[f"test_{metrica}"].mean() for metrica in metricas
    }
    resultados[nome]["desvio_padrao_f1"] = scores["test_f1"].std()
    print(f"  concluído.")


Rodando cross-validation: Regressao Logistica...
  concluído.
Rodando cross-validation: Random Forest...
  concluído.


## 5. Comparar os resultados

In [5]:
tabela_comparacao = pd.DataFrame(resultados).T.round(4)
tabela_comparacao


,accuracy,precision,recall,f1,roc_auc,desvio_padrao_f1
Regressao Logistica,0.6852,0.6525,0.8267,0.7294,0.7566,0.0004
Random Forest,0.6865,0.6685,0.7718,0.7164,0.7598,0.0006


**Como ler:**
- `roc_auc` é a métrica mais informativa aqui — mede a capacidade do
  modelo de separar as duas classes independente do limiar de decisão,
  e não é distorcida pelo balanceamento das classes.
- `desvio_padrao_f1` mostra o quanto o F1 variou entre os 3 folds — um
  desvio pequeno indica que o modelo é estável (não depende de sorte na
  divisão dos dados).


## 6. Escolher o modelo

*(Complete esta célula depois de rodar — a conclusão depende dos números
reais que saírem na sua máquina, então está deixada como decisão aberta
aqui.)*

Critério de escolha: maior `roc_auc` médio, com desvio-padrão baixo o
suficiente para confiar que a diferença não é ruído entre folds.


In [6]:
melhor_modelo = tabela_comparacao["roc_auc"].idxmax()
print(f"Modelo escolhido para a Parte 5: {melhor_modelo}")
tabela_comparacao.loc[[melhor_modelo]]


Modelo escolhido para a Parte 5: Random Forest


,accuracy,precision,recall,f1,roc_auc,desvio_padrao_f1
Random Forest,0.6865,0.6685,0.7718,0.7164,0.7598,0.0006


## 7. Salvar a comparação

Guardamos a tabela em `reports/` — vai direto para o README final como
evidência da comparação de modelos.


In [7]:
PASTA_REPORTS = PROJECT_ROOT / "reports"
PASTA_REPORTS.mkdir(exist_ok=True)

tabela_comparacao.to_csv(PASTA_REPORTS / "comparacao_modelos.csv")
print(f"Salvo em: {PASTA_REPORTS / 'comparacao_modelos.csv'}")


Salvo em: c:\Users\Guilherme\Desktop\Projeto\modelagem\reports\comparacao_modelos.csv
